# DAC 2026 V4 — Official-Data-Only NR@5% Consensus + Local Risk Reranking

**Runtime target:** Kaggle Notebook with **T4 x2**.  
**Official leaderboard metric:** `NormalizedRecall@5%` only.

This V4 is the post-reset continuation of the V1→V3 evidence chain, with the architecture revised for the latest DAC 2026 rules:

- only the organizer-provided `train.csv`, `test.csv`, and `sample_submission.csv` are read;
- `claim_id` is an identifier only and never becomes a predictive feature;
- no previous submissions, cached predictions, helper ground-truth files, external datasets, pretrained external models, or leaderboard-derived row edits are consumed;
- the known source-row label block is treated as a storage-order defect, so folds and model fitting are explicitly order-safe;
- administrative codes are treated as categorical values exactly as required by the updated data dictionary; code-prefix features are intentionally disabled;
- CatBoost remains the global anchor from V1–V3, while XGBoost and LightGBM provide complementary ranking views;
- the V4-specific upgrade is a **multi-scale empirical-Bayes / structured-retrieval risk layer** built only from each outer training fold;
- the top-risk consensus core is preserved, while only a narrow band around the 5% audit boundary may be conservatively reordered;
- deployment uses the same five-fold bagged ranking recipe represented by validation.

The notebook is intentionally **NR@5%-first**. AP, Brier, 3/5/7% allocation, and fairness diagnostics are retained for the analytical report and operational guardrails, but they do not replace the official leaderboard objective.


In [1]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import platform
import random
import sys
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scipy.special import expit
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

try:
    import torch
    GPU_COUNT = int(torch.cuda.device_count())
except Exception:
    GPU_COUNT = 0

IS_KAGGLE = Path('/kaggle').exists()
WORKING = Path('/kaggle/working') if IS_KAGGLE else Path('/tmp')
ARTIFACT_DIR = WORKING / 'v4_artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORKING / 'submission_v4.csv'
ANCHOR_SUBMISSION_PATH = WORKING / 'submission_v4_anchor.csv'

@dataclass(frozen=True)
class Config:
    seed: int = 2026
    n_folds: int = 5
    audit_capacity: float = 0.05
    cat_seed_a: int = 2127
    cat_seed_b: int = 2026
    cat_a_iterations: int = 2860
    cat_b_iterations: int = 2400
    xgb_estimators: int = 1900
    lgbm_estimators: int = 1900
    ohe_min_frequency: int = 20
    eb_alphas: tuple = (1.0, 3.0, 10.0, 30.0)
    frequency_columns: tuple = ('kdkc','dati2','geo_facility','diag_facility','diag_cmg','cmg_facility')
    boundary_bands: tuple = ((0.015,0.075),(0.020,0.080),(0.020,0.090))
    boundary_weights: tuple = (0.20,0.30,0.40)
    min_boundary_gain_tp: int = 4
    min_boundary_fold_wins: int = 4
    max_boundary_fold_tp_loss: int = 2
    min_global_fold_wins: int = 3
    max_global_fold_tp_loss: int = 4

CFG = Config()
random.seed(CFG.seed)
np.random.seed(CFG.seed)

CAT_DEVICES = '0:1' if GPU_COUNT >= 2 else ('0' if GPU_COUNT == 1 else None)
XGB_DEVICE = 'cuda' if GPU_COUNT >= 1 else 'cpu'

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('GPUs visible:', GPU_COUNT)
print('CatBoost devices:', CAT_DEVICES or 'CPU')
print('XGBoost device:', XGB_DEVICE)
print('Artifacts:', ARTIFACT_DIR)


Python: 3.12.13
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
GPUs visible: 2
CatBoost devices: 0:1
XGBoost device: cuda
Artifacts: /kaggle/working/v4_artifacts


## 1. Exact official metric and deterministic label-neutral top-k

The organizer ranks claims by `fraud_probability` and audits exactly `floor(0.05N)` rows. For the balanced development sample used here, NormalizedRecall@5% reduces to the fraction of fraud claims inside that exact top-K portfolio. Ties are resolved only with a deterministic **identifier key**, never with source-row position.


In [2]:
ID_COL = 'claim_id'
TARGET_COL = 'label'


def _binary_target(y):
    y = np.asarray(y)
    if y.ndim != 1:
        raise ValueError('Target must be 1-dimensional.')
    u = set(pd.unique(y).tolist())
    if not u.issubset({0,1}):
        raise ValueError(f'Expected binary organizer labels 0/1; found {sorted(u)}')
    return y.astype(np.int8)


def top_k_mask(scores, capacity=0.05, tie_key=None):
    s = np.asarray(scores, dtype=float)
    if s.ndim != 1 or not np.isfinite(s).all():
        raise ValueError('Scores must be a finite 1-D array.')
    n = len(s)
    k = int(math.floor(float(capacity) * n))
    out = np.zeros(n, dtype=bool)
    if k <= 0:
        return out
    if tie_key is None:
        raise ValueError('A label-neutral tie key is required.')
    key = np.asarray(pd.Series(tie_key).astype('string'))
    if len(key) != n:
        raise ValueError('tie_key length mismatch')
    # lexsort uses the last key as primary: descending score, then identifier string.
    order = np.lexsort((key, -s))
    out[order[:k]] = True
    return out


def normalized_recall_at_capacity(y_true, scores, capacity=0.05, tie_key=None):
    y = _binary_target(y_true)
    selected = top_k_mask(scores, capacity=capacity, tie_key=tie_key)
    n = len(y)
    k = int(math.floor(capacity * n))
    p = int(y.sum())
    if p == 0 or k == 0:
        return 0.0
    tp = int(y[selected].sum())
    recall = tp / p
    recall_max = min(1.0, k / p)
    return float(recall / recall_max) if recall_max > 0 else 0.0


def tp_at_capacity(y_true, scores, capacity=0.05, tie_key=None):
    y = _binary_target(y_true)
    selected = top_k_mask(scores, capacity=capacity, tie_key=tie_key)
    return int(y[selected].sum()), int(selected.sum())


def fold_risk_percentile(values):
    """Higher value = higher risk; strictly label-free."""
    s = pd.Series(np.asarray(values, dtype=float))
    return s.rank(method='average', pct=True).to_numpy(dtype=np.float64)


def foldwise_percentile(values, fold_id):
    out = np.empty(len(values), dtype=np.float64)
    for f in np.unique(fold_id):
        m = np.asarray(fold_id) == f
        out[m] = fold_risk_percentile(np.asarray(values)[m])
    return out

# Metric smoke tests, including the label-block storage-order hazard.
_y = np.array([1,0,1,0,1,0,1,0,1,0] * 10, dtype=int)
_s = np.linspace(0,1,len(_y))
_k = np.array([f'ID_{i:04d}' for i in range(len(_y))])
assert top_k_mask(_s, 0.05, _k).sum() == 5
assert 0.0 <= normalized_recall_at_capacity(_y, _s, 0.05, _k) <= 1.0
assert top_k_mask(np.ones(len(_y)), 0.05, _k).sum() == 5
print('Official NR@5% / exact-K selector checks passed.')


Official NR@5% / exact-K selector checks passed.


## 2. Official-data-only discovery and schema contract

The latest rules prohibit external data. This notebook searches only for a directory containing the organizer's three CSV roles and deliberately rejects directories that look like historical results, notebooks, helper labels, or extracted old-data workspaces. If several valid copies exist, the current competition-style path is preferred deterministically.


In [3]:
SEMANTIC_DTYPES = {ID_COL:'string', 'kdkc':'string', 'dati2':'string'}
FORBIDDEN_PATH_TOKENS = (
    'ground_truth', 'results', 'result-v', 'notebook', 'old_dac', 'old-dac',
    'reference', 'submission_v1', 'submission_v2', 'submission_v3'
)


def sha256_file(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def forbidden_parent(parent: Path):
    text = str(parent).lower()
    return any(tok in text for tok in FORBIDDEN_PATH_TOKENS)


def candidate_triplets():
    roots = [Path('/kaggle/input')] if IS_KAGGLE else [Path('/mnt/data'), Path('.')]
    found = []
    for root in roots:
        if not root.exists():
            continue
        for train_path in root.rglob('train.csv'):
            parent = train_path.parent
            if forbidden_parent(parent):
                continue
            test_path = parent / 'test.csv'
            sample_path = parent / 'sample_submission.csv'
            if not (test_path.exists() and sample_path.exists()):
                continue
            try:
                tr_cols = pd.read_csv(train_path, nrows=2).columns.tolist()
                te_cols = pd.read_csv(test_path, nrows=2).columns.tolist()
                sa_cols = pd.read_csv(sample_path, nrows=2).columns.tolist()
            except Exception:
                continue
            valid = (
                ID_COL in tr_cols and TARGET_COL in tr_cols and
                ID_COL in te_cols and TARGET_COL not in te_cols and
                sa_cols == [ID_COL, 'fraud_probability']
            )
            if valid:
                ptxt = str(parent).lower()
                score = 0
                score += 20 if 'preliminary-round-dac-2026' in ptxt else 0
                score += 10 if 'new-dac-2026' in ptxt or 'new_dac' in ptxt else 0
                score -= len(parent.parts) * 0.01
                found.append((score, parent, train_path, test_path, sample_path))
    return sorted(found, key=lambda x: (-x[0], str(x[1])))


triplets = candidate_triplets()
if not triplets:
    raise FileNotFoundError('Could not find an official train/test/sample_submission triplet.')

_, DATA_DIR, TRAIN_PATH, TEST_PATH, SAMPLE_PATH = triplets[0]
print('Selected official dataset directory:', DATA_DIR)

train_raw = pd.read_csv(TRAIN_PATH, dtype=SEMANTIC_DTYPES)
test_raw = pd.read_csv(TEST_PATH, dtype=SEMANTIC_DTYPES)
sample_submission = pd.read_csv(SAMPLE_PATH, dtype={ID_COL:'string'})

if list(sample_submission.columns) != [ID_COL, 'fraud_probability']:
    raise ValueError('Unexpected sample submission schema.')
if TARGET_COL not in train_raw or TARGET_COL in test_raw:
    raise ValueError('Unexpected target placement.')
if set(train_raw.columns) - {TARGET_COL} != set(test_raw.columns):
    raise ValueError('Train/test feature schema mismatch.')
if train_raw[ID_COL].duplicated().any() or test_raw[ID_COL].duplicated().any():
    raise ValueError('Duplicate claim_id detected.')
if not sample_submission[ID_COL].astype('string').reset_index(drop=True).equals(test_raw[ID_COL].astype('string').reset_index(drop=True)):
    raise ValueError('Sample submission order does not match test.csv claim_id order.')

y = _binary_target(pd.to_numeric(train_raw[TARGET_COL], errors='raise').to_numpy())

DATA_HASHES = {
    'train_sha256': sha256_file(TRAIN_PATH),
    'test_sha256': sha256_file(TEST_PATH),
    'sample_sha256': sha256_file(SAMPLE_PATH),
}

print('train:', train_raw.shape, 'test:', test_raw.shape, 'sample:', sample_submission.shape)
print('fraud prevalence:', float(y.mean()))
print('hashes:', DATA_HASHES)


Selected official dataset directory: /kaggle/input/competitions/dac-its-2026
train: (160174, 53) test: (40043, 52) sample: (40043, 2)
fraud prevalence: 0.5007242124189943
hashes: {'train_sha256': 'e87c42e644e07745e1fdf270949a438e4d636e1497bbcc48b1bea032279ddc3b', 'test_sha256': '4df0cfa840ecf44bc2c6aa09270261f63e939341d276c7b75df1bde4f636b36e', 'sample_sha256': '13d43ce8d76ef78c5cd66a60514db51a492fd58c048c0abee41969dda1e225b2'}


## 3. Data audit and explicit source-order neutralization

The organizer CSV currently stores labels in a highly structured order. That is **not** treated as a feature. We record the defect for reproducibility, then make the modeling contract explicit: no source index, row number, or `claim_id` enters the feature matrix; grouped folds are shuffled; and every fitted model receives a seeded permutation of its training rows.


In [4]:
label_transitions = int(np.sum(y[1:] != y[:-1]))
raw_missing = int(train_raw.isna().sum().sum() + test_raw.isna().sum().sum())

DATA_AUDIT = {
    'train_rows': int(len(train_raw)),
    'test_rows': int(len(test_raw)),
    'train_columns': int(train_raw.shape[1]),
    'test_columns': int(test_raw.shape[1]),
    'fraud_prevalence': float(y.mean()),
    'label_transitions_in_source_order': label_transitions,
    'raw_missing_cells_train_plus_test': raw_missing,
    'train_duplicate_claim_id': int(train_raw[ID_COL].duplicated().sum()),
    'test_duplicate_claim_id': int(test_raw[ID_COL].duplicated().sum()),
}

with open(ARTIFACT_DIR / 'data_audit.json', 'w') as f:
    json.dump({**DATA_AUDIT, **DATA_HASHES}, f, indent=2)

display(pd.DataFrame([DATA_AUDIT]))
if label_transitions <= 2:
    display(Markdown('**Order-safety warning:** labels are strongly blocked in source order. V4 never uses source order as model information.'))


,train_rows,test_rows,train_columns,test_columns,fraud_prevalence,label_transitions_in_source_order,raw_missing_cells_train_plus_test,train_duplicate_claim_id,test_duplicate_claim_id
0,160174,40043,53,52,0.500724,1,0,0,0


**Order-safety warning:** labels are strongly blocked in source order. V4 never uses source order as model information.

## 4. Clean semantic typing and V1–V3 evidence-backed feature package

The feature package retains the useful burden, concentration, rarity, and limited interaction hypotheses from V1–V3, while removing the now-unjustified administrative **prefix** features. `kdkc` and `dati2` remain full categorical codes, as specified by the updated data dictionary.


In [5]:
RAW_CATEGORICAL = ['kdkc','dati2','typeppk','jkpst','jnspelsep','cmg','diagprimer']
DX_COLS = [c for c in train_raw.columns if c.startswith('dx2_')]
PROC_COLS = [c for c in train_raw.columns if c.startswith('proc')]


def clean_claims(frame):
    df = frame.copy()
    df[ID_COL] = df[ID_COL].astype('string')
    for c in RAW_CATEGORICAL:
        if c in df:
            df[c] = df[c].astype('string').str.strip().fillna('__MISSING__')
    for c in ['umur','los','severitylevel', *DX_COLS, *PROC_COLS]:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors='raise')
    return df


def safe_divide(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return np.divide(a, b, out=np.zeros_like(a, dtype=np.float64), where=np.abs(b) > 1e-12)


def block_entropy_share(df, cols):
    x = df[cols].to_numpy(dtype=np.float64)
    total = x.sum(axis=1)
    nonzero = (x > 0).sum(axis=1)
    maxv = x.max(axis=1) if x.shape[1] else np.zeros(len(df))
    share = safe_divide(maxv, total)
    probs = safe_divide(x, total[:, None])
    with np.errstate(divide='ignore', invalid='ignore'):
        ent = -(np.where(probs > 0, probs * np.log(probs), 0.0)).sum(axis=1)
    return total, nonzero, maxv, share, ent


def engineer_features(frame):
    df = clean_claims(frame)
    out = df.copy()

    out['age_band'] = pd.cut(
        out['umur'], bins=[-np.inf,17,39,59,np.inf],
        labels=['0_17','18_39','40_59','60_plus']
    ).astype('string')
    out['is_pediatric'] = (out['umur'] <= 17).astype(np.int8)
    out['is_older_adult'] = (out['umur'] >= 60).astype(np.int8)
    out['log1p_los'] = np.log1p(out['los'].clip(lower=0))
    out['los_zero'] = (out['los'] == 0).astype(np.int8)

    dx_total, dx_nz, dx_max, dx_share, dx_ent = block_entropy_share(out, DX_COLS)
    pr_total, pr_nz, pr_max, pr_share, pr_ent = block_entropy_share(out, PROC_COLS)

    out['diagnosis_total'] = dx_total
    out['diagnosis_nonzero_groups'] = dx_nz
    out['diagnosis_max_group'] = dx_max
    out['diagnosis_max_share'] = dx_share
    out['diagnosis_entropy'] = dx_ent
    out['procedure_total'] = pr_total
    out['procedure_nonzero_groups'] = pr_nz
    out['procedure_max_group'] = pr_max
    out['procedure_max_share'] = pr_share
    out['procedure_entropy'] = pr_ent
    out['coded_burden_total'] = dx_total + pr_total
    out['log1p_coded_burden'] = np.log1p(out['coded_burden_total'])
    denom_days = np.maximum(out['los'].to_numpy(dtype=float), 1.0)
    out['diagnosis_per_day'] = dx_total / denom_days
    out['procedure_per_day'] = pr_total / denom_days
    out['burden_per_day'] = (dx_total + pr_total) / denom_days
    out['procedure_to_diagnosis'] = safe_divide(pr_total, dx_total + 1.0)
    out['total_nonzero_groups'] = dx_nz + pr_nz
    out['age_x_loglos'] = out['umur'].to_numpy(dtype=float) * out['log1p_los'].to_numpy(dtype=float)

    # Limited categorical interactions retained from V1–V3 evidence.
    out['geo_facility'] = out['dati2'].astype(str) + '__' + out['typeppk'].astype(str)
    out['kc_facility'] = out['kdkc'].astype(str) + '__' + out['typeppk'].astype(str)
    out['diag_cmg'] = out['diagprimer'].astype(str) + '__' + out['cmg'].astype(str)
    out['diag_service'] = out['diagprimer'].astype(str) + '__' + out['jnspelsep'].astype(str)
    out['diag_facility'] = out['diagprimer'].astype(str) + '__' + out['typeppk'].astype(str)
    out['cmg_facility'] = out['cmg'].astype(str) + '__' + out['typeppk'].astype(str)
    out['geo_cmg'] = out['dati2'].astype(str) + '__' + out['cmg'].astype(str)
    out['geo_service'] = out['dati2'].astype(str) + '__' + out['jnspelsep'].astype(str)
    out['cmg_severity'] = out['cmg'].astype(str) + '__' + out['severitylevel'].astype('Int64').astype(str)

    return out


train_model = engineer_features(train_raw)
test_model = engineer_features(test_raw)

CONSTANT_COLUMNS = [
    c for c in train_model.columns
    if c not in {ID_COL, TARGET_COL} and train_model[c].nunique(dropna=False) <= 1
]
FEATURE_COLUMNS = [
    c for c in train_model.columns
    if c not in {ID_COL, TARGET_COL, *CONSTANT_COLUMNS}
]
CATEGORICAL_COLUMNS = [
    c for c in FEATURE_COLUMNS
    if pd.api.types.is_string_dtype(train_model[c].dtype) or train_model[c].dtype == 'object'
]
NUMERIC_COLUMNS = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL_COLUMNS]

assert ID_COL not in FEATURE_COLUMNS and TARGET_COL not in FEATURE_COLUMNS
assert not any('prefix' in c.lower() for c in FEATURE_COLUMNS)

print('Raw DX columns:', len(DX_COLS), 'Raw procedure columns:', len(PROC_COLS))
print('Constant excluded:', CONSTANT_COLUMNS)
print('Features:', len(FEATURE_COLUMNS), '| categorical:', len(CATEGORICAL_COLUMNS), '| numeric:', len(NUMERIC_COLUMNS))


Raw DX columns: 22 Raw procedure columns: 19
Constant excluded: ['dx2_koo_k93', 'dx2_u00_u99', 'procv00_v89']
Features: 80 | categorical: 17 | numeric: 63


## 5. Leakage-safe primary validation

Exact repeated cleaned feature signatures stay inside one fold. `StratifiedGroupKFold(..., shuffle=True)` prevents the source-order label block from defining the folds. This is the same core validation protection that survived the V1/V2 forensic audit.


In [6]:
# Signature excludes claim_id and label. Hashing is used only as a grouping key.
signature_frame = clean_claims(train_raw).drop(columns=[ID_COL, TARGET_COL], errors='ignore')
GROUP_SIGNATURE = pd.util.hash_pandas_object(signature_frame, index=False).to_numpy(dtype=np.uint64)

sgkf = StratifiedGroupKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
FOLD_ID = np.full(len(train_model), -1, dtype=np.int8)
FOLDS = []
for fold, (fit_idx, val_idx) in enumerate(sgkf.split(train_model, y, groups=GROUP_SIGNATURE)):
    FOLD_ID[val_idx] = fold
    FOLDS.append((fit_idx, val_idx))

assert (FOLD_ID >= 0).all()
fold_rows=[]
for fold,(fit_idx,val_idx) in enumerate(FOLDS):
    overlap = len(set(GROUP_SIGNATURE[fit_idx]).intersection(set(GROUP_SIGNATURE[val_idx])))
    fold_rows.append({
        'fold':fold,
        'train_rows':len(fit_idx),
        'valid_rows':len(val_idx),
        'valid_fraud_rate':float(y[val_idx].mean()),
        'signature_overlap':overlap,
    })
    if overlap:
        raise AssertionError('Feature-signature leakage across folds.')
FOLD_TABLE = pd.DataFrame(fold_rows)
FOLD_TABLE.to_csv(ARTIFACT_DIR/'fold_definition.csv', index=False)
display(FOLD_TABLE)


,fold,train_rows,valid_rows,valid_fraud_rate,signature_overlap
0,0,128125,32049,0.497987,0
1,1,128064,32110,0.500280,0
2,2,128248,31926,0.502976,0
3,3,128160,32014,0.500031,0
4,4,128099,32075,0.502354,0


## 6. Fold-safe label-free rarity and multi-scale local empirical risk

Two different structures are used:

1. **frequency / rarity features** are label-free and may be supplied to the global models;
2. **empirical-risk retrieval features** use labels and are therefore built only from the outer training fold and applied to its held-out fold/test.

The local-risk layer implements the V4 blueprint directly: multiple hierarchical views, several shrinkage strengths, support, entropy/purity, and Wilson uncertainty. Stronger hierarchies receive fixed prior weights based on the V1–V3 boundary analysis, not on hidden test outcomes.


In [7]:
HIERARCHIES = [
    ('dati2','typeppk','cmg','diagprimer'),
    ('dati2','typeppk','cmg','severitylevel'),
    ('dati2','typeppk','jnspelsep','cmg'),
    ('dati2','typeppk','jnspelsep','diagprimer'),
    ('dati2','typeppk','severitylevel','diagprimer'),
    ('kdkc','typeppk','cmg','diagprimer'),
    ('typeppk','cmg','diagprimer','severitylevel'),
    ('typeppk','jnspelsep','cmg','diagprimer'),
    # Reduced clinical template: official features only.
    ('typeppk','jnspelsep','cmg','severitylevel','diagprimer',
     'diagnosis_total','diagnosis_nonzero_groups','procedure_total','procedure_nonzero_groups'),
]
HIERARCHY_WEIGHTS = np.array([1.40,1.20,1.10,1.05,1.05,0.95,0.90,0.90,1.10], dtype=float)


def fold_frequency_features(fit_df, apply_df, columns):
    out = pd.DataFrame(index=apply_df.index)
    n = max(len(fit_df), 1)
    for c in columns:
        freq = fit_df[c].value_counts(dropna=False) / n
        out[f'freq__{c}'] = apply_df[c].map(freq).fillna(0.0).astype(np.float32)
    return out


def wilson_interval(pos, count, z=1.96):
    pos = np.asarray(pos, dtype=float)
    count = np.asarray(count, dtype=float)
    p = np.divide(pos, count, out=np.zeros_like(pos), where=count > 0)
    denom = 1.0 + z*z/np.maximum(count,1.0)
    center = (p + z*z/(2*np.maximum(count,1.0))) / denom
    rad = z*np.sqrt((p*(1-p)/np.maximum(count,1.0)) + z*z/(4*np.maximum(count,1.0)**2)) / denom
    low = np.where(count > 0, np.clip(center-rad,0,1), np.nan)
    high = np.where(count > 0, np.clip(center+rad,0,1), np.nan)
    return low, high


def hierarchy_statistics(fit_df, fit_y, apply_df, cols, alphas):
    prior = float(np.mean(fit_y))
    work = fit_df[list(cols)].copy()
    work['_target_'] = np.asarray(fit_y, dtype=np.int8)
    stats = work.groupby(list(cols), dropna=False, observed=True)['_target_'].agg(['sum','count']).reset_index()
    merged = apply_df[list(cols)].merge(stats, on=list(cols), how='left', sort=False)
    count = merged['count'].fillna(0).to_numpy(dtype=float)
    pos = merged['sum'].fillna(0).to_numpy(dtype=float)
    raw = np.divide(pos, count, out=np.full_like(pos, prior), where=count > 0)
    low, high = wilson_interval(pos, count)
    low = np.where(np.isfinite(low), low, prior)
    high = np.where(np.isfinite(high), high, prior)

    prefix = 'eb__' + '__'.join(cols)
    out = pd.DataFrame(index=apply_df.index)
    for a in alphas:
        out[f'{prefix}__a{int(a)}'] = ((pos + a*prior)/(count+a)).astype(np.float32)
    out[f'{prefix}__raw'] = raw.astype(np.float32)
    out[f'{prefix}__log_support'] = np.log1p(count).astype(np.float32)
    out[f'{prefix}__purity'] = (np.abs(raw-0.5)*2.0).astype(np.float32)
    out[f'{prefix}__wilson_low'] = low.astype(np.float32)
    out[f'{prefix}__wilson_high'] = high.astype(np.float32)
    return out, count, prior


def local_risk_bundle(fit_df, fit_y, apply_df):
    all_stats=[]
    risk_terms=[]
    rel_terms=[]
    for h_idx, cols in enumerate(HIERARCHIES):
        stats, count, prior = hierarchy_statistics(fit_df, fit_y, apply_df, cols, CFG.eb_alphas)
        all_stats.append(stats)
        prefix = 'eb__' + '__'.join(cols)
        # Conservative multi-scale risk: strong low-alpha signal tempered by Wilson lower bound.
        r = (
            0.35*stats[f'{prefix}__a3'].to_numpy(float) +
            0.30*stats[f'{prefix}__a10'].to_numpy(float) +
            0.20*stats[f'{prefix}__a30'].to_numpy(float) +
            0.15*stats[f'{prefix}__wilson_low'].to_numpy(float)
        )
        reliability = count / (count + 10.0)
        risk_terms.append(r)
        rel_terms.append(reliability * HIERARCHY_WEIGHTS[h_idx])

    full = pd.concat(all_stats, axis=1)
    risk_mat = np.column_stack(risk_terms)
    rel_mat = np.column_stack(rel_terms)
    denom = rel_mat.sum(axis=1)
    prior = float(np.mean(fit_y))
    local = np.divide((risk_mat*rel_mat).sum(axis=1), denom, out=np.full(len(apply_df),prior,dtype=float), where=denom>0)
    full['local_empirical_risk'] = local.astype(np.float32)
    full['local_total_reliability'] = denom.astype(np.float32)
    full['local_risk_min'] = risk_mat.min(axis=1).astype(np.float32)
    full['local_risk_max'] = risk_mat.max(axis=1).astype(np.float32)
    full['local_risk_spread'] = (risk_mat.max(axis=1)-risk_mat.min(axis=1)).astype(np.float32)
    return full

print('Hierarchical retrieval views:', len(HIERARCHIES))
print('Shrinkage strengths:', CFG.eb_alphas)


Hierarchical retrieval views: 9
Shrinkage strengths: (1.0, 3.0, 10.0, 30.0)


## 7. Global model helpers

The roster is intentionally small and evidence-led:

- **CatBoost A**: V2/V3-style depth-8 anchor;
- **CatBoost B**: a slightly shallower, independently seeded CatBoost view;
- **XGBoost** and **LightGBM**: one-hot complementary tree families.

No target encoding is fed into the base learners. That keeps the outer OOF predictions genuinely held out and avoids second-level target-statistic contamination. Local risk is combined later through a fixed, fold-safe ranking rule.


In [8]:
def model_frame(base_df, freq_df):
    x = base_df[FEATURE_COLUMNS].reset_index(drop=True).copy()
    f = freq_df.reset_index(drop=True)
    return pd.concat([x, f], axis=1)


def catboost_ready(df, categorical):
    x = df.copy()
    for c in categorical:
        x[c] = x[c].astype('string').fillna('__MISSING__').astype(str)
    return x


def seeded_order(n, seed):
    return np.random.default_rng(seed).permutation(n)


def fit_catboost(x_fit, y_fit, x_val, x_test, seed, iterations, depth):
    cats = [c for c in CATEGORICAL_COLUMNS if c in x_fit.columns]
    x_fit = catboost_ready(x_fit, cats)
    x_val = catboost_ready(x_val, cats)
    x_test = catboost_ready(x_test, cats)
    order = seeded_order(len(x_fit), seed)
    params = dict(
        iterations=iterations,
        learning_rate=0.03,
        depth=depth,
        loss_function='Logloss',
        eval_metric='Logloss',
        l2_leaf_reg=8.0 if depth >= 8 else 7.0,
        random_strength=0.35,
        bootstrap_type='Bayesian',
        bagging_temperature=0.5,
        max_ctr_complexity=2,
        border_count=128,
        random_seed=seed,
        allow_writing_files=False,
        verbose=False,
        thread_count=-1,
    )
    if GPU_COUNT:
        params.update(task_type='GPU', devices=CAT_DEVICES)
    model = CatBoostClassifier(**params)
    model.fit(x_fit.iloc[order], np.asarray(y_fit)[order], cat_features=cats)
    return (
        model.predict_proba(x_val)[:,1].astype(np.float64),
        model.predict_proba(x_test)[:,1].astype(np.float64),
        model,
    )


def make_sparse_matrices(x_fit, x_val, x_test):
    cats = [c for c in CATEGORICAL_COLUMNS if c in x_fit.columns]
    nums = [c for c in x_fit.columns if c not in cats]
    enc = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=CFG.ohe_min_frequency, sparse_output=True, dtype=np.float32), cats),
            ('num', 'passthrough', nums),
        ],
        sparse_threshold=0.1,
    )
    a = enc.fit_transform(x_fit)
    b = enc.transform(x_val)
    c = enc.transform(x_test)
    return a, b, c, enc


def fit_xgb(x_fit, y_fit, x_val, x_test, seed):
    order = seeded_order(x_fit.shape[0], seed)
    model = XGBClassifier(
        n_estimators=CFG.xgb_estimators,
        learning_rate=0.035,
        max_depth=7,
        min_child_weight=6,
        subsample=0.85,
        colsample_bytree=0.82,
        reg_alpha=0.08,
        reg_lambda=7.0,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        device=XGB_DEVICE,
        random_state=seed,
        n_jobs=-1,
        max_bin=256,
    )
    model.fit(x_fit[order], np.asarray(y_fit)[order], verbose=False)
    return model.predict_proba(x_val)[:,1].astype(float), model.predict_proba(x_test)[:,1].astype(float), model


def fit_lgbm(x_fit, y_fit, x_val, x_test, seed):
    order = seeded_order(x_fit.shape[0], seed)
    model = LGBMClassifier(
        n_estimators=CFG.lgbm_estimators,
        learning_rate=0.025,
        num_leaves=48,
        min_child_samples=60,
        subsample=0.85,
        colsample_bytree=0.82,
        reg_alpha=0.10,
        reg_lambda=7.0,
        objective='binary',
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )
    model.fit(x_fit[order], np.asarray(y_fit)[order])
    return model.predict_proba(x_val)[:,1].astype(float), model.predict_proba(x_test)[:,1].astype(float), model


## 8. Five-fold training, structured retrieval, and checkpointing

Each outer fold independently learns:

- label-free frequency mappings;
- four global model predictions;
- empirical-risk retrieval statistics from the outer training labels only.

The fold's test predictions and local-risk statistics are stored separately so the final deployment can average **complete fold ranking recipes**, not a single full-data refit.


In [9]:
MODEL_NAMES = ['cat_a','cat_b','xgb','lgbm']
OOF_PROBS = {m: np.zeros(len(train_model), dtype=np.float64) for m in MODEL_NAMES}
TEST_PROBS_BY_FOLD = {m: [] for m in MODEL_NAMES}
LOCAL_OOF = np.zeros(len(train_model), dtype=np.float64)
LOCAL_TEST_BY_FOLD = []
LOCAL_DIAGNOSTIC_OOF = np.zeros((len(train_model), 4), dtype=np.float32)
FOLD_MODEL_METRICS=[]

for fold, (fit_idx, val_idx) in enumerate(FOLDS):
    print(f'\n===== OUTER FOLD {fold+1}/{CFG.n_folds} =====')
    fit_base = train_model.iloc[fit_idx].copy()
    val_base = train_model.iloc[val_idx].copy()
    tst_base = test_model.copy()
    y_fit = y[fit_idx]

    freq_fit = fold_frequency_features(fit_base, fit_base, CFG.frequency_columns)
    freq_val = fold_frequency_features(fit_base, val_base, CFG.frequency_columns)
    freq_test = fold_frequency_features(fit_base, tst_base, CFG.frequency_columns)

    x_fit = model_frame(fit_base, freq_fit)
    x_val = model_frame(val_base, freq_val)
    x_test = model_frame(tst_base, freq_test)

    # CatBoost anchor and diverse CatBoost view.
    pva, pta, _ = fit_catboost(x_fit, y_fit, x_val, x_test, CFG.cat_seed_a + fold*31, CFG.cat_a_iterations, 8)
    pvb, ptb, _ = fit_catboost(x_fit, y_fit, x_val, x_test, CFG.cat_seed_b + fold*31, CFG.cat_b_iterations, 7)
    OOF_PROBS['cat_a'][val_idx] = pva
    OOF_PROBS['cat_b'][val_idx] = pvb
    TEST_PROBS_BY_FOLD['cat_a'].append(pta)
    TEST_PROBS_BY_FOLD['cat_b'].append(ptb)

    # Shared one-hot matrices for XGBoost + LightGBM.
    sx_fit, sx_val, sx_test, _ = make_sparse_matrices(x_fit, x_val, x_test)
    pvx, ptx, _ = fit_xgb(sx_fit, y_fit, sx_val, sx_test, CFG.seed + 1000 + fold)
    pvl, ptl, _ = fit_lgbm(sx_fit, y_fit, sx_val, sx_test, CFG.seed + 2000 + fold)
    OOF_PROBS['xgb'][val_idx] = pvx
    OOF_PROBS['lgbm'][val_idx] = pvl
    TEST_PROBS_BY_FOLD['xgb'].append(ptx)
    TEST_PROBS_BY_FOLD['lgbm'].append(ptl)

    # Fold-safe structured retrieval / empirical risk.
    local_val = local_risk_bundle(fit_base, y_fit, val_base)
    local_test = local_risk_bundle(fit_base, y_fit, tst_base)
    LOCAL_OOF[val_idx] = local_val['local_empirical_risk'].to_numpy(float)
    LOCAL_TEST_BY_FOLD.append(local_test['local_empirical_risk'].to_numpy(float))
    LOCAL_DIAGNOSTIC_OOF[val_idx,0] = local_val['local_total_reliability'].to_numpy(float)
    LOCAL_DIAGNOSTIC_OOF[val_idx,1] = local_val['local_risk_min'].to_numpy(float)
    LOCAL_DIAGNOSTIC_OOF[val_idx,2] = local_val['local_risk_max'].to_numpy(float)
    LOCAL_DIAGNOSTIC_OOF[val_idx,3] = local_val['local_risk_spread'].to_numpy(float)

    for name, pred in [('cat_a',pva),('cat_b',pvb),('xgb',pvx),('lgbm',pvl),('local_risk',LOCAL_OOF[val_idx])]:
        tp,k = tp_at_capacity(y[val_idx], pred, CFG.audit_capacity, train_model.iloc[val_idx][ID_COL])
        FOLD_MODEL_METRICS.append({
            'fold':fold,'model':name,'tp_at_5pct':tp,'k':k,
            'nr_at_5pct':normalized_recall_at_capacity(y[val_idx], pred, CFG.audit_capacity, train_model.iloc[val_idx][ID_COL])
        })

    np.savez_compressed(
        ARTIFACT_DIR / f'fold_{fold}_checkpoint.npz',
        val_idx=val_idx,
        cat_a_val=pva, cat_a_test=pta,
        cat_b_val=pvb, cat_b_test=ptb,
        xgb_val=pvx, xgb_test=ptx,
        lgbm_val=pvl, lgbm_test=ptl,
        local_val=LOCAL_OOF[val_idx], local_test=LOCAL_TEST_BY_FOLD[-1],
    )
    del sx_fit, sx_val, sx_test, x_fit, x_val, x_test, fit_base, val_base, tst_base
    gc.collect()

FOLD_MODEL_METRICS = pd.DataFrame(FOLD_MODEL_METRICS)
FOLD_MODEL_METRICS.to_csv(ARTIFACT_DIR/'base_fold_metrics.csv', index=False)
display(FOLD_MODEL_METRICS.pivot(index='fold',columns='model',values='nr_at_5pct'))



===== OUTER FOLD 1/5 =====

===== OUTER FOLD 2/5 =====

===== OUTER FOLD 3/5 =====

===== OUTER FOLD 4/5 =====

===== OUTER FOLD 5/5 =====


model,cat_a,cat_b,lgbm,local_risk,xgb
fold,,,,,
0,0.969413,0.971910,0.961298,0.946317,0.956305
1,0.968224,0.965732,0.953271,0.952648,0.948910
2,0.973684,0.973684,0.963659,0.956140,0.958647
3,0.976250,0.971875,0.962500,0.960000,0.948125
4,0.972551,0.974423,0.961323,0.960075,0.950717


## 9. Fixed global consensus, selected by NR@5% stability

V3's learned stacker improved offline ranking but transferred less strongly than its OOF gain suggested. V4 therefore removes the extra learned meta layer and uses a compact family of **fixed rank-consensus formulas**. Every ingredient is already outer-fold held out. This is more conservative and directly reproducible on test.


In [10]:
# Normalize each model to a fold-local risk percentile before blending.
OOF_RANKS = {m: foldwise_percentile(OOF_PROBS[m], FOLD_ID) for m in MODEL_NAMES}
LOCAL_RANK_OOF = foldwise_percentile(LOCAL_OOF, FOLD_ID)

TEST_RANKS_BY_FOLD = {
    m: [fold_risk_percentile(v) for v in TEST_PROBS_BY_FOLD[m]]
    for m in MODEL_NAMES
}
LOCAL_RANK_TEST_BY_FOLD = [fold_risk_percentile(v) for v in LOCAL_TEST_BY_FOLD]

GLOBAL_FORMULAS = {
    'cat_a': {'cat_a':1.00},
    'cat_mean': {'cat_a':0.55,'cat_b':0.45},
    'consensus_55_20_15_10': {'cat_a':0.55,'cat_b':0.20,'lgbm':0.15,'xgb':0.10},
    'consensus_50_20_20_10': {'cat_a':0.50,'cat_b':0.20,'lgbm':0.20,'xgb':0.10},
    'consensus_60_15_15_10': {'cat_a':0.60,'cat_b':0.15,'lgbm':0.15,'xgb':0.10},
}


def weighted_score(ranks, weights):
    out = np.zeros(len(next(iter(ranks.values()))), dtype=float)
    for name,w in weights.items():
        out += float(w) * np.asarray(ranks[name], dtype=float)
    return out


def score_fold_metrics(score, reference=None):
    rows=[]
    for fold, (_, val_idx) in enumerate(FOLDS):
        tp,k = tp_at_capacity(y[val_idx], score[val_idx], CFG.audit_capacity, train_model.iloc[val_idx][ID_COL])
        row={'fold':fold,'tp':tp,'k':k,'nr':normalized_recall_at_capacity(y[val_idx],score[val_idx],CFG.audit_capacity,train_model.iloc[val_idx][ID_COL])}
        if reference is not None:
            tp0,_=tp_at_capacity(y[val_idx],reference[val_idx],CFG.audit_capacity,train_model.iloc[val_idx][ID_COL])
            row['delta_tp']=tp-tp0
        rows.append(row)
    return pd.DataFrame(rows)

GLOBAL_CANDIDATES={}
GLOBAL_REGISTRY=[]
anchor_score = weighted_score(OOF_RANKS, GLOBAL_FORMULAS['cat_a'])
anchor_tp, anchor_k = tp_at_capacity(y, anchor_score, CFG.audit_capacity, train_model[ID_COL])

for name,weights in GLOBAL_FORMULAS.items():
    s = weighted_score(OOF_RANKS, weights)
    tp,k = tp_at_capacity(y,s,CFG.audit_capacity,train_model[ID_COL])
    fm = score_fold_metrics(s, anchor_score)
    wins = int((fm['delta_tp']>0).sum()) if name!='cat_a' else 0
    worst = int(fm['delta_tp'].min()) if name!='cat_a' else 0
    eligible = (name=='cat_a') or (wins >= CFG.min_global_fold_wins and worst >= -CFG.max_global_fold_tp_loss)
    GLOBAL_CANDIDATES[name]=s
    GLOBAL_REGISTRY.append({
        'candidate':name,'tp_at_5pct':tp,'k':k,
        'nr_at_5pct':normalized_recall_at_capacity(y,s,CFG.audit_capacity,train_model[ID_COL]),
        'delta_tp_vs_cat_a':tp-anchor_tp,'fold_wins_vs_cat_a':wins,'worst_fold_delta_tp':worst,
        'eligible':bool(eligible),
    })

GLOBAL_REGISTRY=pd.DataFrame(GLOBAL_REGISTRY)
elig=GLOBAL_REGISTRY[GLOBAL_REGISTRY['eligible']].copy()
elig=elig.sort_values(['tp_at_5pct','worst_fold_delta_tp','fold_wins_vs_cat_a'],ascending=[False,False,False])
SELECTED_GLOBAL=str(elig.iloc[0]['candidate'])
GLOBAL_OOF=GLOBAL_CANDIDATES[SELECTED_GLOBAL]

# Fold-specific test score for the selected fixed formula.
SELECTED_WEIGHTS=GLOBAL_FORMULAS[SELECTED_GLOBAL]
GLOBAL_TEST_BY_FOLD=[]
for fold in range(CFG.n_folds):
    fold_ranks={m:TEST_RANKS_BY_FOLD[m][fold] for m in MODEL_NAMES}
    GLOBAL_TEST_BY_FOLD.append(weighted_score(fold_ranks,SELECTED_WEIGHTS))

GLOBAL_REGISTRY['selected'] = GLOBAL_REGISTRY['candidate'].eq(SELECTED_GLOBAL)
GLOBAL_REGISTRY.to_csv(ARTIFACT_DIR/'global_candidate_registry.csv',index=False)
display(GLOBAL_REGISTRY.sort_values('tp_at_5pct',ascending=False))
print('Selected global consensus:',SELECTED_GLOBAL,SELECTED_WEIGHTS)


,candidate,tp_at_5pct,k,nr_at_5pct,delta_tp_vs_cat_a,fold_wins_vs_cat_a,worst_fold_delta_tp,eligible,selected
1,cat_mean,7795,8008,0.973402,11,4,-2,True,True
0,cat_a,7784,8008,0.972028,0,0,0,True,False
2,consensus_55_20_15_10,7769,8008,0.970155,-15,1,-8,False,False
4,consensus_60_15_15_10,7766,8008,0.969780,-18,1,-10,False,False
3,consensus_50_20_20_10,7765,8008,0.969655,-19,1,-9,False,False


Selected global consensus: cat_mean {'cat_a': 0.55, 'cat_b': 0.45}


## 10. Consensus-core preservation and hard-negative local reranking

The V3 refiner was too aggressive: it could replace the entire ordering across a broad top-10% candidate pool. V4 instead **freezes membership** outside a narrow boundary band and only reorders rows already inside that band. This means:

- the extreme high-consensus core is not displaced;
- claims well below the audit boundary cannot jump into the portfolio;
- the local empirical-risk layer acts as conservative hard-negative evidence around the 5% cutoff;
- boundary weight is capped at 0.40, never 1.00.

The small predeclared grid is selected by pooled NR@5% plus fold-win / worst-fold safeguards.


In [11]:
def rerank_band_one(score, local_score, lower_top, upper_top, weight):
    """Reorder only rows whose original rank lies between lower_top and upper_top fractions from the top."""
    score=np.asarray(score,dtype=float).copy()
    local_score=np.asarray(local_score,dtype=float)
    pct=fold_risk_percentile(score)  # high risk -> near 1
    top_fraction=1.0-pct
    mask=(top_fraction >= lower_top) & (top_fraction <= upper_top)
    idx=np.flatnonzero(mask)
    if len(idx) < 2:
        return score
    g_rank=fold_risk_percentile(score[idx])
    l_rank=fold_risk_percentile(local_score[idx])
    combined=(1.0-weight)*g_rank + weight*l_rank
    original_values=np.sort(score[idx])
    order=np.argsort(combined,kind='mergesort')
    reassigned=np.empty(len(idx),dtype=float)
    reassigned[order]=original_values
    score[idx]=reassigned
    return score


def rerank_oof_foldwise(global_score, local_score, lower_top, upper_top, weight):
    out=np.asarray(global_score,dtype=float).copy()
    for fold,(_,val_idx) in enumerate(FOLDS):
        out[val_idx]=rerank_band_one(out[val_idx],local_score[val_idx],lower_top,upper_top,weight)
    return out


def topk_churn(a,b,capacity=0.05):
    ma=top_k_mask(a,capacity,train_model[ID_COL])
    mb=top_k_mask(b,capacity,train_model[ID_COL])
    return int(np.logical_xor(ma,mb).sum()//2)

base_tp,_=tp_at_capacity(y,GLOBAL_OOF,CFG.audit_capacity,train_model[ID_COL])
BOUNDARY_CANDIDATES={'global_only':GLOBAL_OOF}
BOUNDARY_REGISTRY=[]

# Reference row.
base_fm=score_fold_metrics(GLOBAL_OOF,GLOBAL_OOF)
BOUNDARY_REGISTRY.append({
    'candidate':'global_only','lower_top':np.nan,'upper_top':np.nan,'weight':0.0,
    'tp_at_5pct':base_tp,'delta_tp':0,'fold_wins':0,'worst_fold_delta_tp':0,
    'top5_churn':0,'eligible':True,
})

for lower,upper in CFG.boundary_bands:
    for weight in CFG.boundary_weights:
        name=f'band_{lower:.3f}_{upper:.3f}_w{weight:.2f}'
        s=rerank_oof_foldwise(GLOBAL_OOF,LOCAL_OOF,lower,upper,weight)
        tp,_=tp_at_capacity(y,s,CFG.audit_capacity,train_model[ID_COL])
        fm=score_fold_metrics(s,GLOBAL_OOF)
        wins=int((fm['delta_tp']>0).sum())
        worst=int(fm['delta_tp'].min())
        churn=topk_churn(GLOBAL_OOF,s,CFG.audit_capacity)
        eligible=(
            tp-base_tp >= CFG.min_boundary_gain_tp and
            wins >= CFG.min_boundary_fold_wins and
            worst >= -CFG.max_boundary_fold_tp_loss
        )
        BOUNDARY_CANDIDATES[name]=s
        BOUNDARY_REGISTRY.append({
            'candidate':name,'lower_top':lower,'upper_top':upper,'weight':weight,
            'tp_at_5pct':tp,'delta_tp':tp-base_tp,'fold_wins':wins,'worst_fold_delta_tp':worst,
            'top5_churn':churn,'eligible':bool(eligible),
        })

BOUNDARY_REGISTRY=pd.DataFrame(BOUNDARY_REGISTRY)
elig=BOUNDARY_REGISTRY[BOUNDARY_REGISTRY['eligible']].copy()
# If several candidates are within 3 TP of the best, prefer lower weight, narrower band, and less churn.
if len(elig):
    best_tp=int(elig['tp_at_5pct'].max())
    near=elig[elig['tp_at_5pct'] >= best_tp-3].copy()
    near['band_width']=(near['upper_top']-near['lower_top']).fillna(0.0)
    near=near.sort_values(['worst_fold_delta_tp','fold_wins','weight','band_width','top5_churn'],ascending=[False,False,True,True,True])
    SELECTED_BOUNDARY=str(near.iloc[0]['candidate'])
else:
    SELECTED_BOUNDARY='global_only'

FINAL_OOF=BOUNDARY_CANDIDATES[SELECTED_BOUNDARY]
BOUNDARY_REGISTRY['selected']=BOUNDARY_REGISTRY['candidate'].eq(SELECTED_BOUNDARY)
BOUNDARY_REGISTRY.to_csv(ARTIFACT_DIR/'boundary_candidate_registry.csv',index=False)
display(BOUNDARY_REGISTRY.sort_values('tp_at_5pct',ascending=False))
print('Selected boundary recipe:',SELECTED_BOUNDARY)


,candidate,lower_top,upper_top,weight,tp_at_5pct,delta_tp,fold_wins,worst_fold_delta_tp,top5_churn,eligible,selected
0,global_only,NaN,NaN,0.0,7795,0,0,0,0,True,True
8,band_0.020_0.090_w0.30,0.020,0.090,0.3,7787,-8,2,-5,491,False,False
5,band_0.020_0.080_w0.30,0.020,0.080,0.3,7786,-9,1,-5,423,False,False
2,band_0.015_0.075_w0.30,0.015,0.075,0.3,7786,-9,1,-5,405,False,False
4,band_0.020_0.080_w0.20,0.020,0.080,0.2,7784,-11,0,-6,248,False,False
1,band_0.015_0.075_w0.20,0.015,0.075,0.2,7783,-12,0,-6,225,False,False
7,band_0.020_0.090_w0.20,0.020,0.090,0.2,7783,-12,1,-6,284,False,False
3,band_0.015_0.075_w0.40,0.015,0.075,0.4,7781,-14,1,-9,581,False,False
6,band_0.020_0.080_w0.40,0.020,0.080,0.4,7780,-15,1,-9,616,False,False
9,band_0.020_0.090_w0.40,0.020,0.090,0.4,7773,-22,0,-7,712,False,False


Selected boundary recipe: global_only


## 11. Fold-bagged test deployment

The selected boundary recipe is now applied **independently to each outer fold's test predictions**, using that fold's own local-risk mapping. Five complete rank recipes are then averaged. There is no V1-style single full-data refit and no V3-style one-shot refiner mismatch.


In [12]:
if SELECTED_BOUNDARY == 'global_only':
    FINAL_TEST_BY_FOLD=[np.asarray(x,dtype=float) for x in GLOBAL_TEST_BY_FOLD]
else:
    row=BOUNDARY_REGISTRY.loc[BOUNDARY_REGISTRY['candidate'].eq(SELECTED_BOUNDARY)].iloc[0]
    lower=float(row['lower_top']); upper=float(row['upper_top']); weight=float(row['weight'])
    FINAL_TEST_BY_FOLD=[]
    for fold in range(CFG.n_folds):
        FINAL_TEST_BY_FOLD.append(
            rerank_band_one(GLOBAL_TEST_BY_FOLD[fold],LOCAL_TEST_BY_FOLD[fold],lower,upper,weight)
        )

# Rank-average complete fold recipes to avoid probability-scale differences between fold models.
FINAL_TEST_FOLD_RANKS=np.vstack([fold_risk_percentile(v) for v in FINAL_TEST_BY_FOLD])
FINAL_TEST_SCORE=FINAL_TEST_FOLD_RANKS.mean(axis=0)
GLOBAL_TEST_SCORE=np.vstack([fold_risk_percentile(v) for v in GLOBAL_TEST_BY_FOLD]).mean(axis=0)

# OOF score is also normalized foldwise before calibration/diagnostics.
FINAL_OOF_SCORE=foldwise_percentile(FINAL_OOF,FOLD_ID)
GLOBAL_OOF_SCORE=foldwise_percentile(GLOBAL_OOF,FOLD_ID)

fold_final_rows=[]
for fold,(_,val_idx) in enumerate(FOLDS):
    tp,k=tp_at_capacity(y[val_idx],FINAL_OOF_SCORE[val_idx],CFG.audit_capacity,train_model.iloc[val_idx][ID_COL])
    tp0,_=tp_at_capacity(y[val_idx],GLOBAL_OOF_SCORE[val_idx],CFG.audit_capacity,train_model.iloc[val_idx][ID_COL])
    fold_final_rows.append({'fold':fold,'tp':tp,'k':k,'delta_tp_vs_global':tp-tp0,'nr_at_5pct':normalized_recall_at_capacity(y[val_idx],FINAL_OOF_SCORE[val_idx],CFG.audit_capacity,train_model.iloc[val_idx][ID_COL])})
FINAL_FOLD_METRICS=pd.DataFrame(fold_final_rows)
FINAL_FOLD_METRICS.to_csv(ARTIFACT_DIR/'final_fold_metrics.csv',index=False)
display(FINAL_FOLD_METRICS)


,fold,tp,k,delta_tp_vs_global,nr_at_5pct
0,0,1557,1602,0,0.971910
1,1,1556,1605,0,0.969470
2,2,1558,1596,0,0.976190
3,3,1560,1600,0,0.975000
4,4,1562,1603,0,0.974423


## 12. OOF diagnostics: exact 3%, 5%, and 7% audit portfolios

Only NR@5% selects the leaderboard model. The additional statistics are retained because the official analytical report still asks participants to understand capacity sensitivity, unnecessary legitimate audits, probability reliability, and policyholder protection.


In [13]:
def age_group(age):
    age=np.asarray(age,dtype=float)
    return np.select([age<=17,age<=39,age<=59],['0-17','18-39','40-59'],default='60+')


def protection_score(y_true, selected, gender, age, min_nonfraud=100):
    yb=_binary_target(y_true)
    selected=np.asarray(selected,dtype=bool)
    gender=np.asarray(pd.Series(gender).astype(str))
    ageg=age_group(age)
    gaps=[]
    details=[]
    for attr,groups in [('gender',gender),('age',ageg)]:
        fprs=[]
        for g in pd.unique(groups):
            m=(groups==g)&(yb==0)
            denom=int(m.sum())
            if denom < min_nonfraud:
                continue
            num=int((selected&m).sum())
            fpr=num/denom
            fprs.append(fpr)
            details.append({'attribute':attr,'group':g,'nonfraud_n':denom,'legitimate_audits':num,'fpr':fpr})
        gaps.append(max(fprs)-min(fprs) if len(fprs)>=2 else 0.0)
    return 1.0-max(gaps or [0.0]), pd.DataFrame(details)

AUDIT_ROWS=[]
FAIRNESS_ROWS=[]
for cap in [0.03,0.05,0.07]:
    selected=top_k_mask(FINAL_OOF_SCORE,cap,train_model[ID_COL])
    tp=int(y[selected].sum()); k=int(selected.sum()); legit=k-tp
    protection,detail=protection_score(y,selected,train_model['jkpst'],train_model['umur'])
    AUDIT_ROWS.append({
        'capacity':cap,'k':k,'fraud_captured':tp,'legitimate_audits':legit,
        'normalized_recall':normalized_recall_at_capacity(y,FINAL_OOF_SCORE,cap,train_model[ID_COL]),
        'portfolio_precision':tp/max(k,1),'protection_score':protection,
    })
    detail['capacity']=cap
    FAIRNESS_ROWS.append(detail)

AUDIT_TABLE=pd.DataFrame(AUDIT_ROWS)
FAIRNESS_TABLE=pd.concat(FAIRNESS_ROWS,ignore_index=True)
AUDIT_TABLE.to_csv(ARTIFACT_DIR/'audit_allocation.csv',index=False)
FAIRNESS_TABLE.to_csv(ARTIFACT_DIR/'fairness_detail.csv',index=False)
display(AUDIT_TABLE)

# Global diagnostics that do not drive model selection.
FINAL_AP=float(average_precision_score(y,FINAL_OOF_SCORE))
FINAL_TP,FINAL_K=tp_at_capacity(y,FINAL_OOF_SCORE,CFG.audit_capacity,train_model[ID_COL])
GLOBAL_TP,_=tp_at_capacity(y,GLOBAL_OOF_SCORE,CFG.audit_capacity,train_model[ID_COL])
print('Final pooled TP@5%:',FINAL_TP,'/',FINAL_K,'NR@5%=',normalized_recall_at_capacity(y,FINAL_OOF_SCORE,CFG.audit_capacity,train_model[ID_COL]))
print('Global pooled TP@5%:',GLOBAL_TP,'delta=',FINAL_TP-GLOBAL_TP)
print('OOF AP diagnostic:',FINAL_AP)


,capacity,k,fraud_captured,legitimate_audits,normalized_recall,portfolio_precision,protection_score
0,0.03,4805,4718,87,0.981894,0.981894,0.999260
1,0.05,8008,7795,213,0.973402,0.973402,0.998600
2,0.07,11212,10789,423,0.962273,0.962273,0.997245


Final pooled TP@5%: 7795 / 8008 NR@5%= 0.9734015984015985
Global pooled TP@5%: 7795 delta= 0
OOF AP diagnostic: 0.8247506127103857


## 13. Rank-preserving probability map

The current leaderboard uses ranking only, but the required column is still named `fraud_probability`. V4 therefore fits a single scalar logistic map from the final OOF rank score to the observed train labels. A positive coefficient guarantees that calibration cannot change the final ordering.


In [14]:
calibrator=LogisticRegression(C=1e6,solver='lbfgs',max_iter=2000,random_state=CFG.seed)
calibrator.fit(FINAL_OOF_SCORE.reshape(-1,1),y)
coef=float(calibrator.coef_[0,0]); intercept=float(calibrator.intercept_[0])
if not np.isfinite(coef) or coef <= 0:
    raise RuntimeError(f'Rank-preserving calibration failed: coefficient={coef}')

OOF_PROB=expit(intercept + coef*FINAL_OOF_SCORE)
TEST_PROB=expit(intercept + coef*FINAL_TEST_SCORE)
GLOBAL_TEST_PROB=expit(intercept + coef*GLOBAL_TEST_SCORE)

print('Calibration coefficient:',coef,'intercept:',intercept)
print('OOF Brier diagnostic:',float(brier_score_loss(y,OOF_PROB)))
print('Probability range test:',float(TEST_PROB.min()),float(TEST_PROB.max()))


Calibration coefficient: 4.929071715895219 intercept: -2.4605674843595606
OOF Brier diagnostic: 0.17259348833038804
Probability range test: 0.07868882635449471 0.9218899610741401


## 14. Submission construction and strict validation

The primary file is the evidence-selected V4 recipe. An anchor file is also saved for diagnostic/submission-slot comparison, but the notebook labels the selected primary unambiguously. Both use the organizer's exact test ID order.


In [15]:
def validate_submission(submission, sample):
    if list(submission.columns) != [ID_COL,'fraud_probability']:
        raise ValueError('Submission columns must be exactly claim_id,fraud_probability')
    if len(submission) != len(sample):
        raise ValueError('Submission row count mismatch')
    if submission[ID_COL].duplicated().any():
        raise ValueError('Duplicate claim_id in submission')
    if not submission[ID_COL].astype('string').reset_index(drop=True).equals(sample[ID_COL].astype('string').reset_index(drop=True)):
        raise ValueError('Submission claim_id order mismatch')
    p=pd.to_numeric(submission['fraud_probability'],errors='coerce').to_numpy(float)
    if not np.isfinite(p).all() or (p<0).any() or (p>1).any():
        raise ValueError('Invalid fraud_probability values')
    return True

submission=pd.DataFrame({ID_COL:sample_submission[ID_COL].astype('string'),'fraud_probability':TEST_PROB})
anchor_submission=pd.DataFrame({ID_COL:sample_submission[ID_COL].astype('string'),'fraud_probability':GLOBAL_TEST_PROB})
validate_submission(submission,sample_submission)
validate_submission(anchor_submission,sample_submission)
submission.to_csv(SUBMISSION_PATH,index=False)
anchor_submission.to_csv(ANCHOR_SUBMISSION_PATH,index=False)

print('PRIMARY:',SUBMISSION_PATH)
print('ANCHOR :',ANCHOR_SUBMISSION_PATH)
print('Rows:',len(submission))
display(submission.head())


PRIMARY: /kaggle/working/submission_v4.csv
ANCHOR : /kaggle/working/submission_v4_anchor.csv
Rows: 40043


,claim_id,fraud_probability
0,CLM_8D29A008FB3F6B,0.730669
1,CLM_E15AA2EC5E05DB,0.916706
2,CLM_71FDA31CB6D5B0,0.825896
3,CLM_E478EDD63AA6DB,0.874042
4,CLM_F5407EE803152B,0.825428


## 15. Reproducibility manifest and run summary

The manifest records exactly which official files, ranking recipe, fold configuration, and model settings produced the submission. No historical V1–V3 prediction artifact is needed to reproduce V4.


In [16]:
MANIFEST={
    'version':'DAC2026_V4_OfficialOnly_NR5',
    'official_metric':'NormalizedRecall@5%',
    'data_dir':str(DATA_DIR),
    **DATA_HASHES,
    'config':asdict(CFG),
    'gpu_count':GPU_COUNT,
    'selected_global':SELECTED_GLOBAL,
    'selected_global_weights':SELECTED_WEIGHTS,
    'selected_boundary':SELECTED_BOUNDARY,
    'oof_tp_at_5pct':int(FINAL_TP),
    'oof_k_at_5pct':int(FINAL_K),
    'oof_nr_at_5pct':float(normalized_recall_at_capacity(y,FINAL_OOF_SCORE,CFG.audit_capacity,train_model[ID_COL])),
    'oof_ap_diagnostic':float(FINAL_AP),
    'calibration_coef':coef,
    'calibration_intercept':intercept,
    'submission_path':str(SUBMISSION_PATH),
    'submission_sha256':sha256_file(SUBMISSION_PATH),
    'external_data_used':False,
    'historical_prediction_artifacts_used':False,
    'claim_id_used_as_predictor':False,
    'source_row_used_as_predictor':False,
    'administrative_prefix_features_used':False,
}
with open(ARTIFACT_DIR/'manifest.json','w') as f:
    json.dump(MANIFEST,f,indent=2,default=str)

np.savez_compressed(
    ARTIFACT_DIR/'selected_predictions.npz',
    oof_score=FINAL_OOF_SCORE,
    oof_probability=OOF_PROB,
    global_oof_score=GLOBAL_OOF_SCORE,
    test_score=FINAL_TEST_SCORE,
    test_probability=TEST_PROB,
    global_test_score=GLOBAL_TEST_SCORE,
    fold_id=FOLD_ID,
)

with open(ARTIFACT_DIR/'config.json','w') as f:
    json.dump(asdict(CFG),f,indent=2,default=str)

display(Markdown(
    f"""### V4 run complete

- **Selected global recipe:** `{SELECTED_GLOBAL}`
- **Selected boundary recipe:** `{SELECTED_BOUNDARY}`
- **OOF TP@5%:** `{FINAL_TP}/{FINAL_K}`
- **OOF NR@5%:** `{normalized_recall_at_capacity(y,FINAL_OOF_SCORE,CFG.audit_capacity,train_model[ID_COL]):.6f}`
- **Primary submission:** `{SUBMISSION_PATH}`
- **Submission SHA-256:** `{MANIFEST['submission_sha256']}`

The primary file is generated solely from the current official train/test data and the code in this notebook.
"""
))


### V4 run complete

- **Selected global recipe:** `cat_mean`
- **Selected boundary recipe:** `global_only`
- **OOF TP@5%:** `7795/8008`
- **OOF NR@5%:** `0.973402`
- **Primary submission:** `/kaggle/working/submission_v4.csv`
- **Submission SHA-256:** `9ad43290ac5ed461e3110f08722ca141b53714692aa70c84b3a0d1336f98d3ed`

The primary file is generated solely from the current official train/test data and the code in this notebook.


## 16. Interpretation boundary for the analytical report

The final ranking is a **decision-support priority score**, not proof of fraudulent intent and not an autonomous rejection rule. The local empirical-risk signals describe recurring patterns in the organizer's historical labels. They may reflect case mix, provider/geographic structure, investigation history, or other data-generation mechanisms; none should be presented as causal evidence.

For the report, preserve the V1–V3 findings that remained valid after the Kaggle reset: exact-signature dependence, strong geographic/facility signal, the importance of audit-capacity analysis, location transport limitations, and human oversight. Replace historical leaderboard claims with results from the reset competition and treat this V4 notebook as the reproducible current prediction pipeline.
